In [ ]:
# Satellite Damage Assessment: Ukraine Conflict Analysis

## Data Cleaning & Feature Engineering

This notebook processes Sentinel-2 satellite imagery from Microsoft Planetary Computer to detect infrastructure damage across Ukrainian conflict zones (Mariupol, Kharkiv, Bakhmut).

**Pipeline:**
1. Download Sentinel-2 Level-2A from Planetary Computer
2. Apply cloud masking
3. Compute spectral indices (NDVI, NDBI, BSI)
4. Save cleaned dataset for modeling

## 1. Setup & Environment

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# Setup paths (notebook runs from ../notebooks/)
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

# ============ SPECTRAL INDICES CLASS ============
class SpectralIndices:
    """Compute spectral indices from Sentinel-2 bands."""
    
    @staticmethod
    def ndvi(image):
        """NDVI = (NIR - RED) / (NIR + RED)"""
        red = image[..., 2].astype(float)
        nir = image[..., 3].astype(float)
        return (nir - red) / (nir + red + 1e-8)
    
    @staticmethod
    def ndbi(image):
        """NDBI = (SWIR - NIR) / (SWIR + NIR)"""
        swir = image[..., 4].astype(float)
        nir = image[..., 3].astype(float)
        return (swir - nir) / (swir + nir + 1e-8)
    
    @staticmethod
    def bsi(image):
        """BSI = (SWIR + RED - NIR - BLUE) / (SWIR + RED + NIR + BLUE)"""
        blue = image[..., 0].astype(float)
        red = image[..., 2].astype(float)
        nir = image[..., 3].astype(float)
        swir = image[..., 4].astype(float)
        numerator = swir + red - nir - blue
        denominator = swir + red + nir + blue
        return numerator / (denominator + 1e-8)

# ============ HELPER FUNCTIONS ============
def apply_cloud_mask(image, scl_band):
    """Apply cloud masking using Scene Classification band."""
    valid_pixels = np.isin(scl_band, [4, 5, 6])
    image_masked = image.copy().astype(float)
    image_masked[~valid_pixels] = np.nan
    return image_masked

def normalize_bands(image):
    """Normalize image bands to [0, 1]."""
    return np.clip(image / 10000.0, 0, 1)

def create_patches(image, patch_size=128, stride=64):
    """Extract overlapping patches from image."""
    h, w = image.shape[:2]
    patches = []
    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patches.append(image[i:i+patch_size, j:j+patch_size])
    return patches

print("✓ Environment ready")

## 2. Download Sentinel-2 Data from Microsoft Planetary Computer

In [ ]:
# Install required packages (run once)
# !pip install pystac-client planetary-computer odc-stac xarray rioxarray

import pystac_client
import planetary_computer
from odc.stac import load
from odc.geo.geom import BoundingBox

print("Connecting to Microsoft Planetary Computer...")

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)

print("✓ Connected to Planetary Computer")

# Define study areas
study_areas = {
    'Mariupol': [37.40, 46.95, 37.68, 47.15],
    'Kharkiv': [35.80, 49.85, 36.50, 50.10],
    'Bakhmut': [37.70, 48.50, 37.90, 48.65]
}

# UPDATED: Quarterly time periods for more data points
periods = {
    '2021_Q1': '2021-01-01/2021-03-31',
    '2021_Q2': '2021-04-01/2021-06-30',
    '2021_Q3': '2021-07-01/2021-09-30',
    '2021_Q4': '2021-10-01/2021-12-31',
    
    '2022_Q1': '2022-01-01/2022-03-31',
    '2022_Q2': '2022-04-01/2022-06-30',
    '2022_Q3': '2022-07-01/2022-09-30',
    '2022_Q4': '2022-10-01/2022-12-31',
    
    '2023_Q1': '2023-01-01/2023-03-31',
    '2023_Q2': '2023-04-01/2023-06-30',
    '2023_Q3': '2023-07-01/2023-09-30',
    '2023_Q4': '2023-10-01/2023-12-31',
    
    '2024_Q1': '2024-01-01/2024-03-31',
    '2024_Q2': '2024-04-01/2024-06-30',
}

print(f"\nDownloading Sentinel-2 data...")
print(f"Cities: {list(study_areas.keys())}")
print(f"Time periods: {len(periods)} quarters")
print(f"Expected scenes: {len(study_areas) * len(periods)}\\n")

## 3. Download and Process Each Scene

In [ ]:
def normalize_for_display(band):
    """Normalize a band to 0-1 for display."""
    band = band.astype(float)
    band = np.nan_to_num(band, nan=0)
    band_min = np.percentile(band[band > 0], 2) if np.any(band > 0) else 0
    band_max = np.percentile(band[band > 0], 98) if np.any(band > 0) else 1
    band_norm = (band - band_min) / (band_max - band_min + 1e-8)
    return np.clip(band_norm, 0, 1)

def create_rgb_image(image, bands_indices=[2, 1, 0]):
    """Create RGB image from specific bands for visualization.
    
    Default: Natural color (R=B04, G=B03, B=B02)
    Alternative: False color (R=B08, G=B04, B=B03) - use [3, 2, 1]
    """
    rgb = np.zeros((image.shape[0], image.shape[1], 3))
    for i, band_idx in enumerate(bands_indices):
        rgb[..., i] = normalize_for_display(image[..., band_idx])
    return (rgb * 255).astype(np.uint8)

print("✓ Visualization functions ready")

In [ ]:
from odc.geo.geom import BoundingBox

metadata = []
downloaded_count = 0
failed_count = 0

for city, bbox in study_areas.items():
    for period_name, datetime_range in periods.items():
        try:
            print(f"Searching {city:10} {period_name}...", end=' ')

            search = catalog.search(
                collections=['sentinel-2-l2a'],
                bbox=bbox,
                datetime=datetime_range,
                query={'eo:cloud_cover': {'lt': 30}}
            )

            items = list(search.get_items())
            if len(items) == 0:
                print("No scenes found")
                continue

            item = items[0]
            print(f"Downloading...", end='')

            # Clip to the study-area bbox — avoids downloading the full 100x100 km tile
            clip_box = BoundingBox(
                left=bbox[0], bottom=bbox[1], right=bbox[2], top=bbox[3],
                crs='epsg:4326'
            )
            data = load(
                [item],
                bands=['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'SCL'],
                resolution=20,
                geopolygon=clip_box,
                patch_url=planetary_computer.sign_url
            )

            # Extract each band properly
            image_array = []
            for band in ['B02', 'B03', 'B04', 'B08', 'B11', 'B12', 'SCL']:
                band_data = data[band].values
                band_data = np.squeeze(band_data)
                image_array.append(band_data)

            image = np.stack(image_array, axis=2)

            filename = f"../data/raw/S2_{period_name}_{city}.npy"
            np.save(filename, image.astype(np.uint16))  # uint16 fits 0-10000 range

            cloud_cover = item.properties.get('eo:cloud_cover', 0)
            metadata.append({
                'filename': filename,
                'city': city,
                'period': period_name,
                'cloud_cover_pct': cloud_cover,
                'scene_id': item.id
            })

            print(f" ✓")
            downloaded_count += 1

        except Exception as e:
            print(f" ✗ {str(e)[:40]}")
            failed_count += 1

print(f"\n{'='*60}")
print(f"Downloaded: {downloaded_count} scenes")
print(f"Failed: {failed_count} scenes")
print(f"{'='*60}")

if metadata:
    df_raw = pd.DataFrame(metadata)
    df_raw.to_csv('../data/raw/metadata.csv', index=False)
    print(f"\nMetadata saved to: ../data/raw/metadata.csv")
    print(df_raw)

## 4. Cloud Masking & Normalization

In [ ]:
# Load metadata
df_raw = pd.read_csv('../data/raw/metadata.csv')

print("Applying cloud mask and normalizing data...\n")

df_cleaned = df_raw.copy()
cleaned_count = 0

for idx, row in df_raw.iterrows():
    try:
        image    = np.load(row['filename'])
        spectral = image[..., :6].astype(np.float32)  # float32 saves 50% vs float64
        scl_band = image[..., 6].astype(int)

        spectral_masked     = apply_cloud_mask(spectral, scl_band)
        spectral_normalized = spectral_masked / 10000.0

        clean_filename = row['filename'].replace('.npy', '_clean.npy')
        np.save(clean_filename, spectral_normalized.astype(np.float32))

        # Delete the raw uint16 file — no longer needed once cleaned
        try:
            os.remove(row['filename'])
        except OSError:
            pass

        valid_ratio = np.isfinite(spectral_normalized).sum() / spectral_normalized.size * 100
        df_cleaned.loc[idx, 'valid_pixels_pct']   = valid_ratio
        df_cleaned.loc[idx, 'cleaned_filename']   = clean_filename

        cleaned_count += 1

    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")
        continue

df_cleaned.to_csv('../data/processed/metadata_cleaned.csv', index=False)

print(f"✓ Cleaned {cleaned_count}/{len(df_raw)} scenes")
print(f"Average valid pixels: {df_cleaned['valid_pixels_pct'].mean():.1f}%")
print(f"\nCleaned metadata saved to: ../data/processed/metadata_cleaned.csv")

## 5. Compute Spectral Indices

In [ ]:
# Load metadata with the new format
df_cleaned = pd.read_csv('../data/processed/metadata_cleaned.csv')

print("Computing spectral indices (NDVI, NDBI, BSI)...\n")

si = SpectralIndices()
indices_data = []

for idx, row in df_cleaned.iterrows():
    try:
        # Load cleaned scene
        image = np.load(row['cleaned_filename'])
        
        # Compute indices
        ndvi = si.ndvi(image)
        ndbi = si.ndbi(image)
        bsi = si.bsi(image)
        
        # Extract year and period from filename
        # Format: ../data/processed/S2_2021_Q1_Mariupol_clean.npy
        filename_parts = row['cleaned_filename'].split('_')
        year = int(filename_parts[1])  # 2021, 2022, etc.
        quarter = filename_parts[2]     # Q1, Q2, Q3, Q4
        period = f"{year}_{quarter}"    # 2021_Q1, 2022_Q2, etc.
        
        # Calculate statistics
        indices_data.append({
            'filename': row['cleaned_filename'],
            'city': row['city'],
            'period': period,
            'year': year,
            'quarter': quarter,
            'cloud_cover_pct': row['cloud_cover_pct'],
            'ndvi_mean': np.nanmean(ndvi),
            'ndvi_std': np.nanstd(ndvi),
            'ndvi_min': np.nanmin(ndvi),
            'ndvi_max': np.nanmax(ndvi),
            'ndbi_mean': np.nanmean(ndbi),
            'ndbi_std': np.nanstd(ndbi),
            'ndbi_min': np.nanmin(ndbi),
            'ndbi_max': np.nanmax(ndbi),
            'bsi_mean': np.nanmean(bsi),
            'bsi_std': np.nanstd(bsi),
            'bsi_min': np.nanmin(bsi),
            'bsi_max': np.nanmax(bsi),
            'valid_pixels_pct': row['valid_pixels_pct']
        })
        
    except Exception as e:
        print(f"Error computing indices for {row['cleaned_filename']}: {e}")
        continue

# Create final dataset
if indices_data:
    df_indices = pd.DataFrame(indices_data)
    df_indices.to_csv('../data/processed/sentinel2_clean_2021_2024.csv', index=False)
    
    print(f"✓ Computed indices for {len(indices_data)} scenes")
    print(f"\nFinal dataset saved to: ../data/processed/sentinel2_clean_2021_2024.csv")
    print(f"\nDataset shape: {df_indices.shape}")
    print(f"\nFirst few rows:")
    print(df_indices.head(10).to_string())
    print(f"\nStatistics:")
    print(df_indices[['ndvi_mean', 'ndbi_mean', 'bsi_mean']].describe().round(3))
else:
    print("ERROR: No data was processed!")
    print("Make sure ../data/processed/metadata_cleaned.csv exists")

## 6. Data Quality Summary

In [ ]:
print("DATA QUALITY SUMMARY")
print("="*70)

print(f"\n1. Temporal Coverage:")
for year in sorted(df_indices['year'].unique()):
    year_data = df_indices[df_indices['year'] == year]
    count = len(year_data)
    quarters = sorted(year_data['quarter'].unique())
    print(f"   {year}: {count} scenes - {', '.join(quarters)}")

print(f"\n2. Geographic Coverage:")
for city in sorted(df_indices['city'].unique()):
    count = (df_indices['city'] == city).sum()
    print(f"   {city}: {count} scenes")

print(f"\n3. Data Quality Metrics:")
print(f"   Average cloud cover: {df_indices['cloud_cover_pct'].mean():.1f}%")
print(f"   Average valid pixels: {df_indices['valid_pixels_pct'].mean():.1f}%")
print(f"   Missing values: {df_indices.isnull().sum().sum()}")

print(f"\n4. Spectral Index Ranges:")
print(f"   NDVI: [{df_indices['ndvi_mean'].min():.3f}, {df_indices['ndvi_mean'].max():.3f}]")
print(f"   NDBI: [{df_indices['ndbi_mean'].min():.3f}, {df_indices['ndbi_mean'].max():.3f}]")
print(f"   BSI:  [{df_indices['bsi_mean'].min():.3f}, {df_indices['bsi_mean'].max():.3f}]")

print(f"\n5. Total Dataset:")
print(f"   Scenes: {len(df_indices)}")
print(f"   Time span: {df_indices['year'].min()} to {df_indices['year'].max()}")
print(f"   Quarters: {df_indices['quarter'].nunique()}")

print(f"\n" + "="*70)
print("✓ Data cleaning complete!")

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for city in df_indices['city'].unique():
    city_data = df_indices[df_indices['city'] == city].sort_values('period')
    
    # Create numeric x-axis for quarters
    x = range(len(city_data))
    
    axes[0, 0].plot(x, city_data['ndvi_mean'], marker='o', label=city, linewidth=2)
    axes[0, 1].plot(x, city_data['ndbi_mean'], marker='s', label=city, linewidth=2)
    axes[1, 0].plot(x, city_data['bsi_mean'], marker='^', label=city, linewidth=2)

# Mark conflict start (2022-02-24 = around early 2022 Q1)
# That's around index 4-5 (after 2021 Q1, Q2, Q3, Q4, and early 2022)
conflict_quarter_index = 4.5  # Between 2021 Q4 and 2022 Q2

for ax in [axes[0, 0], axes[0, 1], axes[1, 0]]:
    ax.axvline(conflict_quarter_index, color='red', linestyle='--', alpha=0.7, linewidth=2, label='Conflict Start')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')

axes[0, 0].set_ylabel('NDVI (Vegetation)', fontsize=11)
axes[0, 0].set_title('Vegetation Index - Quarterly Evolution', fontsize=12, fontweight='bold')

axes[0, 1].set_ylabel('NDBI (Built-up)', fontsize=11)
axes[0, 1].set_title('Built-up Index - Quarterly Evolution', fontsize=12, fontweight='bold')

axes[1, 0].set_ylabel('BSI (Bare Soil)', fontsize=11)
axes[1, 0].set_xlabel('Quarter (0=2021_Q1, 13=2024_Q2)', fontsize=11)
axes[1, 0].set_title('Bare Soil Index - Quarterly Evolution', fontsize=12, fontweight='bold')

axes[1, 1].axis('off')

plt.suptitle('Sentinel-2 Quarterly Spectral Analysis: Ukrainian Conflict Zones', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/spectral_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Visualization saved to: ../results/plots/spectral_analysis.png")
plt.show()

## 8. Summary

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

print("\n" + "="*70)
print("FINAL CLEANED DATASET SUMMARY")
print("="*70)
print(df_indices.to_string(index=False))

print(f"\n✓ Main output: ../data/processed/sentinel2_clean_2021_2024.csv")
print(f"✓ Total scenes: {len(df_indices)}")
print(f"✓ Features: {df_indices.shape[1]}")
print(f"\nNext step: Move to Notebook 2 (Feature Engineering)")
print("="*70)

# Load cleaned images and display them
print("\\nLoading and displaying cleaned Sentinel-2 imagery...\\n")

# Pick one city (e.g., Mariupol)
city_to_display = 'Mariupol'
city_files = [f for f in os.listdir('../data/raw/') 
              if f.startswith(f'S2_') and city_to_display in f and '_clean.npy' in f]
city_files.sort()

print(f"Found {len(city_files)} scenes for {city_to_display}")

# Create a large grid showing all time periods
n_cols = 4
n_rows = (len(city_files) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 12))
axes = axes.flatten()

for idx, filename in enumerate(city_files):
    filepath = os.path.join('../data/raw/', filename)
    image = np.load(filepath)
    
    # Create RGB image (natural color)
    rgb = create_rgb_image(image, bands_indices=[2, 1, 0])
    
    axes[idx].imshow(rgb)
    
    # Extract date from filename (e.g., S2_2021_Q1_Mariupol_clean.npy)
    period = filename.split('_')[1] + '_' + filename.split('_')[2]
    axes[idx].set_title(f'{period}', fontsize=10, fontweight='bold')
    axes[idx].axis('off')

# Hide empty subplots
for idx in range(len(city_files), len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Sentinel-2 Temporal Evolution: {city_to_display}\\n(Natural Color: Red=B04, Green=B03, Blue=B02)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/temporal_grid.png', dpi=150, bbox_inches='tight')
print(f"✓ Temporal grid saved to: ../results/plots/temporal_grid.png")
plt.show()

## 9. Create Animated GIF

In [ ]:
import imageio
from PIL import Image, ImageDraw
from IPython.display import Image as IPImage
import ipywidgets as widgets
from IPython.display import display, HTML

print("Creating animated GIF and interactive viewer...\n")

city_to_animate = 'Mariupol'  # Change to 'Kharkiv' or 'Bakhmut' if desired

# Find all scenes for this city
city_files = [f for f in os.listdir('../data/raw/') 
              if f.startswith(f'S2_') and city_to_animate in f and '_clean.npy' in f]
city_files.sort()

print(f"Creating interactive viewer from {len(city_files)} scenes for {city_to_animate}...\n")

frames = []
frame_labels = []

for filename in city_files:
    filepath = os.path.join('../data/raw/', filename)
    image = np.load(filepath)
    
    # Create RGB image (natural color)
    rgb = create_rgb_image(image, bands_indices=[2, 1, 0])
    
    # Convert to PIL Image
    pil_image = Image.fromarray(rgb)
    
    # Add text with date
    draw = ImageDraw.Draw(pil_image)
    period = filename.split('_')[1] + ' ' + filename.split('_')[2]
    frame_labels.append(period)
    
    # Draw semi-transparent background for text
    draw.rectangle([0, 0, 250, 60], fill=(0, 0, 0, 200))
    draw.text((10, 10), period, fill='white', font=None)
    draw.text((10, 35), f'{city_to_animate}', fill='yellow', font=None)
    
    frames.append(pil_image)

print(f"✓ Prepared {len(frames)} frames\n")

# ============ CREATE SLOWER GIF ============
gif_path = f'../results/plots/{city_to_animate}_temporal_animation.gif'
frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=1500,  # 1500ms (1.5 seconds) per frame - SLOWER!
    loop=0  # Loop forever
)

print(f"✓ GIF saved to: {gif_path}")
print(f"  - Speed: 1.5 seconds per frame")
print(f"  - Total loop time: {len(frames) * 1.5:.1f} seconds")

# ============ INTERACTIVE VIEWER IN NOTEBOOK ============
print(f"\n✓ Interactive Viewer Ready Below!\n")
print("="*70)

# Create interactive slider
frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frames)-1,
    step=1,
    description='Frame:',
    continuous_update=False
)

# Create output widget for image
output = widgets.Output()

# Create info display
info_output = widgets.Output()

def update_frame(frame_idx):
    """Update displayed frame when slider changes."""
    with output:
        output.clear_output(wait=True)
        # Display the frame
        display(frames[frame_idx])
    
    with info_output:
        info_output.clear_output(wait=True)
        period = frame_labels[frame_idx]
        progress = f"Frame {frame_idx + 1}/{len(frames)}"
        
        print(f"Period: {period}")
        print(f"Progress: {progress}")
        print(f"({(frame_idx / len(frames)) * 100:.1f}% through timeline)")
        
        # Add interpretation
        if '2021' in period:
            print("→ PRE-CONFLICT: Healthy vegetation (green areas visible)")
        elif '2022_Q1' in period or '2022_Q2' in period:
            print("→ EARLY CONFLICT (Jan-Jun 2022): Still relatively intact")
        elif '2022' in period:
            print("→ CONFLICT INTENSIFIES (Jul-Dec 2022): Major damage appears")
        elif '2023' in period:
            print("→ ONGOING DAMAGE (2023): Continued destruction, minimal recovery")
        elif '2024' in period:
            print("→ 2+ YEARS INTO CONFLICT: Severe long-term damage visible")

# Create button controls
prev_button = widgets.Button(description='← Previous', button_style='info')
next_button = widgets.Button(description='Next →', button_style='info')
play_button = widgets.ToggleButton(description='▶ Play', button_style='success')

def on_prev_clicked(b):
    if frame_slider.value > 0:
        frame_slider.value -= 1

def on_next_clicked(b):
    if frame_slider.value < len(frames) - 1:
        frame_slider.value += 1

prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Button container
button_box = widgets.HBox([prev_button, play_button, next_button])

# Handle slider changes
def on_slider_change(change):
    update_frame(change['new'])

frame_slider.observe(on_slider_change, names='value')

# Initial display
update_frame(0)

# Display everything
display(HTML("<h3>Sentinel-2 Interactive Time Series Viewer</h3>"))
display(button_box)
display(frame_slider)
display(output)
display(HTML("<hr>"))
display(info_output)

print("\n" + "="*70)
print("Controls:")
print("  • Use the slider to jump to any frame")
print("  • Click '← Previous' / 'Next →' to step through")
print("  • Click '▶ Play' to auto-play (optional)")
print("="*70)